# Mark — Narrative Arc (Morphological Analysis)

Analysis of the narrative arc of the Gospel of Mark from **Greek verbal morphology**,
using the **N1904** dataset (Nestle 1904, CenterBLC) via Text-Fabric.

## What is a Narrative Arc?

A **[narrative arc](https://en.wikipedia.org/wiki/Story_arc)** is the intensity curve of a story over time — the literary equivalent
of an electrocardiogram: where the heart beats fast, action concentrates; where it slows,
the narrative catches its breath, explains, teaches, or meditates.

In a detective novel, the arc rises toward the climax and falls after the revelation.
In a Greek tragedy, it follows Aristotle's five-act structure: exposition → rising tension
→ crisis → fall → resolution. In a sacred text, the arc says something subtler:
**where the text accumulates accomplished events, it *narrates*;
where it accumulates exhortations and questions, it *teaches*.**

What the narrative arc reveals about a text:

- **The author's rhythm**: does Mark write in short bursts of action or long meditations?
- **Hidden structure**: are there narrative turning points that the chapter divisions
  (added in the 13<sup>th</sup> century!) fail to capture?
- **Stylistic singularity**: is Mark really more "staccato" than Matthew or Luke,
  as commentators have claimed for centuries — and can we *measure* it?

## How It Works

Every Greek verb is scored *realis* or *irrealis* based on its **mood** and **tense**:

- **Strong realis** (indicative aorist/perfect): events presented as accomplished facts — *maximum narrative density*
- **Continuous realis** (indicative present/imperfect): ongoing or habitual actions — *narrative background*
- **Background** (participles, indicative pluperfect): circumstances and preconditions
- **Irrealis** (subjunctive, optative, imperative, infinitive, future): potential, commanded, hoped-for — *discourse and exhortation*

The mean score per chapter traces a **narrative arc curve**:
a peak = chapter dense with accomplished action; a trough = discourse chapter.

## Approach vs BookNLP

BookNLP ([Github](https://github.com/booknlp/booknlp)) derives narrative intensity via semantic heuristics (entity density, dialogue).
This approach is **morpho-aspectual**: it relies on the philologically verified data of N1904,
not a predictive model. Theoretical foundations: Porter & Fanning (Greek verbal aspect),
Fleischman (narrative discourse analysis).

## Extension

The same pipeline runs on all 4 gospels → publishable comparative figure
(Mark very aoristic, John dips at ch.13–17 = Farewell Discourse).

In [ ]:
from tf.app import use
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.signal import savgol_filter

N1904 = use("CenterBLC/N1904", version="1.0.0", hoist=globals(), silent=True)

print("✅ N1904 dataset loaded.")

## REALIS_MAP — Classification Table

- Positive scores = realis (event presented as actually accomplished or ongoing)
- Negative scores = irrealis (event that is potential, commanded, future, or conditional)

Exegetical decisions:
- The indicative aorist (1.0) is the pre-eminent marker of narrative reality in Koine Greek:
  it presents an event as a completed point in time.
- The indicative future (−0.3) is classified irrealis despite its indicative mood: it references
  an event not yet realized at the moment of utterance.
- Participles (0.4) are ambiguous: an aorist participle in an "attendant circumstance"
  construction (ἀπεκρίθη εἶπεν) semantically carries the same value as its main verb. This case
  is quantified separately in Section 6.
- The wildcard keys (`mood`, `'*'`) apply to all tenses for moods where tense is
  aspectually secondary (subjunctive, optative, imperative, infinitive, participle).

In [ ]:
REALIS_MAP = {
    # (mood, tense) -> (category, weight)
    ("indicative", "aorist"):     ("realis_strong",    1.0),
    ("indicative", "present"):    ("realis_continuous", 0.7),
    ("indicative", "imperfect"):  ("realis_continuous", 0.6),
    ("indicative", "perfect"):    ("realis_strong",    0.9),
    ("indicative", "pluperfect"): ("realis_background", 0.5),
    ("indicative", "future"):     ("irrealis_future",  -0.3),
    ("subjunctive", "*"):         ("irrealis",         -0.5),
    ("optative", "*"):            ("irrealis",         -0.6),
    ("imperative", "*"):          ("irrealis_command", -0.4),
    ("infinitive", "*"):          ("irrealis_modal",   -0.2),
    ("participle", "*"):          ("realis_background", 0.4),
}

print(f"REALIS_MAP loaded — {len(REALIS_MAP)} entries")

## 1. Morphological Feature Inspection (Mark 1:2–5)

Verification that the N1904 feature names (`mood`, `tense`, `voice`) match
the keys of `REALIS_MAP`. Mark 1:1 contains no verb, so we start at 1:2.

In [ ]:
rows = []
for v_num in range(2, 6):
    verse_node = T.nodeFromSection(('Mark', 1, v_num))
    words = L.d(verse_node, otype='word')
    for w in words:
        if F.sp.v(w) == 'verb' and len(rows) < 10:
            rows.append({
                "Ref":    f"Mk 1:{v_num}",
                "Text":   F.text.v(w),
                "Lemma":  F.lemma.v(w),
                "Gloss":  F.gloss.v(w),
                "Mood":   F.mood.v(w),
                "Tense":  F.tense.v(w),
                "Voice":  F.voice.v(w),
                "Weight": REALIS_MAP.get(
                    (F.mood.v(w), F.tense.v(w)),
                    REALIS_MAP.get((F.mood.v(w), "*"), ("?", "?"))
                )[1],
            })

df_inspect = pd.DataFrame(rows)
display(df_inspect)

## 2. Scoring Functions

In [ ]:
def get_verb_weight(mood: str, tense: str):
    """
    Return (category, weight) for a verb given its mood and tense.
    Tries the exact key (mood, tense) first, then the wildcard (mood, '*').
    Returns None if no entry matches (should not happen with N1904 data).
    """
    key = (mood, tense)
    if key in REALIS_MAP:
        return REALIS_MAP[key]
    wildcard = (mood, "*")
    if wildcard in REALIS_MAP:
        return REALIS_MAP[wildcard]
    return None


def score_chapter(book: str, chapter_num: int) -> dict:
    """
    Compute the realis/irrealis composition and net score for a full chapter.

    Traversal: the chapter node (2-tuple without verse) is more efficient
    than a verse-by-verse loop and yields the same words.

    Normalisation: mean weight over classified verbs only (not total words),
    so chapters are comparable regardless of length.

    Irrealis sub-categories (irrealis_future, irrealis_command, irrealis_modal)
    are collapsed into 'irrealis' for display percentages.
    """
    chapter_node = T.nodeFromSection((book, chapter_num))
    words = L.d(chapter_node, otype='word')

    counts = {
        "realis_strong":     0,
        "realis_continuous": 0,
        "realis_background": 0,
        "irrealis":          0,
    }
    weights = []

    for w in words:
        if F.sp.v(w) != 'verb':
            continue
        result = get_verb_weight(F.mood.v(w), F.tense.v(w))
        if result is None:
            continue
        cat, weight = result
        weights.append(weight)
        # Collapse irrealis sub-categories for display
        display_cat = cat if cat in counts else "irrealis"
        counts[display_cat] += 1

    total = len(weights)
    net_score = sum(weights) / total if total else 0.0

    def pct(k):
        return round(counts[k] / total * 100, 1) if total else 0.0

    return {
        "chapter":               chapter_num,
        "net_score":             round(net_score, 4),
        "verb_count":            total,
        "realis_strong_pct":     pct("realis_strong"),
        "realis_continuous_pct": pct("realis_continuous"),
        "realis_background_pct": pct("realis_background"),
        "irrealis_pct":          pct("irrealis"),
    }


GOSPEL_CHAPTERS = {
    "Mark":     16,
    "Matthew":  28,
    "Luke":     24,
    "John":     21,
}


def build_gospel_df(book: str) -> pd.DataFrame:
    """
    Build a per-chapter realis DataFrame for a given gospel.

    Columns: chapter, net_score, verb_count,
             realis_strong_pct, realis_continuous_pct,
             realis_background_pct, irrealis_pct
    """
    n = GOSPEL_CHAPTERS[book]
    rows = [score_chapter(book, ch) for ch in range(1, n + 1)]
    return pd.DataFrame(rows)


print("✅ Functions defined: get_verb_weight, score_chapter, build_gospel_df")

## 3. Mark DataFrame — Chapter Composition

In [ ]:
df_mark = build_gospel_df("Mark")

# Sanity check: no chapter without verbs
assert df_mark["verb_count"].min() > 0, "Chapter with no verbs detected — check dataset"

max_ch = int(df_mark.loc[df_mark.net_score.idxmax(), "chapter"])
min_ch = int(df_mark.loc[df_mark.net_score.idxmin(), "chapter"])

print(f"Most realis chapter  : ch.{max_ch} (net_score={df_mark.net_score.max():.3f})")
print(f"Least realis chapter : ch.{min_ch} (net_score={df_mark.net_score.min():.3f})")
print()

styled = (
    df_mark.style
    .format({
        "net_score":             "{:.4f}",
        "realis_strong_pct":     "{:.1f}%",
        "realis_continuous_pct": "{:.1f}%",
        "realis_background_pct": "{:.1f}%",
        "irrealis_pct":          "{:.1f}%",
    })
    .background_gradient(subset=["net_score"], cmap="RdYlGn")
    .set_caption("Mark — Verbal Modal Composition by Chapter")
)
display(styled)

## 4. Modal Composition by Chapter (Stacked Bar)

Each bar represents 100% of the classified verbs in that chapter, split across 4 categories.

In [ ]:
COLORS = {
    "realis_strong":     "#1a4f72",
    "realis_continuous": "#5b9bd5",
    "realis_background": "#a0a0a0",
    "irrealis":          "#e07b39",
}

STACK_COLS = [
    "realis_strong_pct",
    "realis_continuous_pct",
    "realis_background_pct",
    "irrealis_pct",
]

STACK_LABELS = [
    "Strong realis (ind. aorist/perfect)",
    "Continuous realis (ind. present/imperfect)",
    "Background (ptc. / ind. pluperfect)",
    "Irrealis (subj./opt./imp./inf./fut.)",
]

chapters = df_mark["chapter"].tolist()
bottom = [0.0] * len(chapters)

fig, ax = plt.subplots(figsize=(14, 6))

for col, label, color in zip(STACK_COLS, STACK_LABELS, COLORS.values()):
    vals = df_mark[col].tolist()
    ax.bar(chapters, vals, bottom=bottom, label=label, color=color, width=0.8)
    bottom = [b + v for b, v in zip(bottom, vals)]

ax.set_xlabel("Chapter", fontsize=12)
ax.set_ylabel("% of classified verbs", fontsize=12)
ax.set_title("Mark — Verbal Modal Composition by Chapter", fontsize=14)
ax.set_xticks(chapters)
ax.legend(loc="upper right", fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

## 5. Smoothed Narrative Arc

The Savitzky-Golay filter (window=3, degree=2) smooths the raw curve while preserving
peaks and troughs — unlike a moving average, which tends to flatten them.

In [ ]:
net_scores = df_mark["net_score"].values
smoothed = savgol_filter(net_scores, window_length=3, polyorder=2)

fig, ax = plt.subplots(figsize=(12, 5))

# Raw scores (faint background)
ax.plot(chapters, net_scores,
        color="#5b9bd5", linewidth=1, linestyle="--", alpha=0.5,
        label="Raw score (per chapter)")

# Smoothed narrative arc
ax.plot(chapters, smoothed,
        color="#1a4f72", linewidth=2.5,
        label="Smoothed arc (Savitzky-Golay, window=3)")

# Annotate extremes
ax.annotate(
    f"Ch.{max_ch}\n(narrative peak)",
    xy=(max_ch, smoothed[max_ch - 1]),
    xytext=(max_ch + 0.6, smoothed[max_ch - 1] + 0.02),
    fontsize=9, arrowprops=dict(arrowstyle="->", color="#333")
)
ax.annotate(
    f"Ch.{min_ch}",
    xy=(min_ch, smoothed[min_ch - 1]),
    xytext=(min_ch - 2.5, smoothed[min_ch - 1] - 0.04),
    fontsize=9, arrowprops=dict(arrowstyle="->", color="#333")
)

# Realis/irrealis boundary
ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")

ax.set_xlabel("Chapter", fontsize=12)
ax.set_ylabel("Net realis score (mean weight)", fontsize=12)
ax.set_title("Mark — Narrative Arc (Greek Verbal Morphology, N1904)", fontsize=14)
ax.set_xticks(chapters)
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 6. Edge Case: Aorist Participle Before a Finite Verb

Mark is rich in constructions of the type **ἀπεκρίθη εἶπεν** (lit. "having answered, he said"),
where an aorist participle functions as an *attendant circumstance* — carrying the same aspectual
value as the main verb that follows.

In `REALIS_MAP`, all participles receive 0.4 (background). This cell quantifies
the number of instances where that score is potentially underestimated.

In [ ]:
def detect_participle_before_main_verb(book: str) -> pd.DataFrame:
    """
    Detect instances of an aorist participle immediately followed by an indicative
    verb in the sequential verb stream of the book.

    Uses the sequential verb list (not L.n()) to avoid the complexity of
    multi-granularity adjacent node tuples.

    Returns a DataFrame: Ref, Participle, Ptc. tense, Main verb, Verb tense, Verb mood
    """
    book_node = T.nodeFromSection((book,))
    all_words = L.d(book_node, otype='word')
    only_verbs = [w for w in all_words if F.sp.v(w) == 'verb']

    rows = []
    for i, w1 in enumerate(only_verbs[:-1]):
        w2 = only_verbs[i + 1]
        if (
            F.mood.v(w1) == 'participle'
            and F.tense.v(w1) == 'aorist'
            and F.mood.v(w2) == 'indicative'
        ):
            ref = T.sectionFromNode(w1)
            rows.append({
                "Ref":        f"{ref[0]} {ref[1]}:{ref[2]}",
                "Participle": F.text.v(w1),
                "Ptc. tense": F.tense.v(w1),
                "Main verb":  F.text.v(w2),
                "Verb tense": F.tense.v(w2),
                "Verb mood":  F.mood.v(w2),
            })

    return pd.DataFrame(rows)


df_ptc = detect_participle_before_main_verb("Mark")
print(f"Aorist participle before indicative verb in Mark: {len(df_ptc)} instances")
print("\nExamples (first 10):")
display(df_ptc.head(10))
print()
print("Note: REALIS_MAP assigns 0.4 to all participles.")
print("These constructions arguably deserve 1.0 (same weight as their main verb).")
print(f"Estimated impact: {len(df_ptc)} under-scored verbs out of {df_mark['verb_count'].sum()} total.")

## 7. Four-Gospel Comparison

The X axis is **normalized (0→1)** to align the curves despite different lengths
(Mark 16 ch., Matthew 28 ch., Luke 24 ch., John 21 ch.).

Computationally verifiable exegetical hypotheses:
- **Mark**: high curve, dense in aorists (staccato narrative style)
- **John**: sharp dip at ch.13–17 (Farewell Discourse: subjunctives, imperatives)
- **Matthew**: peak at ch.5–7 = paradox (Sermon on the Mount = high irrealis)
- **Luke**: intermediate profile (narrator + Acts-style discourse)

In [ ]:
# Build DataFrames for all four gospels (~10 seconds)
dfs = {}
for book in ["Mark", "Matthew", "Luke", "John"]:
    print(f"Processing {book}...", end=" ")
    dfs[book] = build_gospel_df(book)
    print("OK")

GOSPEL_COLORS = {
    "Mark":    "#1a4f72",
    "Matthew": "#e07b39",
    "Luke":    "#2e8b57",
    "John":    "#8b2252",
}

GOSPEL_LABELS = {
    "Mark":    "Mark (16 ch.)",
    "Matthew": "Matthew (28 ch.)",
    "Luke":    "Luke (24 ch.)",
    "John":    "John (21 ch.)",
}

fig, ax = plt.subplots(figsize=(14, 6))

for book, df in dfs.items():
    scores = df["net_score"].values
    n = len(scores)
    # Normalized X axis: narrative position 0→1
    x_norm = [ch / n for ch in range(1, n + 1)]
    smoothed = savgol_filter(scores, window_length=3, polyorder=2)
    ax.plot(x_norm, smoothed,
            label=GOSPEL_LABELS[book],
            color=GOSPEL_COLORS[book],
            linewidth=2)

ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
ax.set_xlabel("Narrative position (0 = beginning, 1 = end)", fontsize=12)
ax.set_ylabel("Smoothed net realis score", fontsize=12)
ax.set_title(
    "Comparative Narrative Arc — The Four Gospels\n(Verbal Morphology, N1904, Nestle 1904)",
    fontsize=14
)
ax.legend(fontsize=11)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 8. (Optional) French BJ Text for Extreme Chapters

The BJ dataset (Bible de Jérusalem) is available locally as a Text-Fabric corpus.
This section displays the French text of the highest- and lowest-scoring chapters
to contextualize the morphological results.

**Note:** The BJ/TOB epub sources live in
`~/Documents/Code/Gemini/antigravity/ScripturesApp/epubs/`.
The conversion scripts (`scripts/converters/convert_bj_epub.py`) can be extended
to export additional structural annotations (section headings, footnotes, pericope titles)
as TF features — enabling richer French-specific analysis (BJ pericope boundaries
mapped onto the Greek morphological arc).

In [ ]:
import os

# BJ book codes for the gospels: MRK, MAT, LUK, JHN
BJ_BOOK = "MRK"
BJ_PATH = os.path.expanduser("~/text-fabric-data/BJ/1.0")

try:
    from tf.fabric import Fabric as TFabric
    TF_BJ = TFabric(locations=BJ_PATH, silent=True)
    api_bj = TF_BJ.load("book chapter verse text", silent=True)
    F_bj, T_bj, L_bj = api_bj.F, api_bj.T, api_bj.L

    def get_bj_chapter_text(book_code: str, chapter_num: int) -> str:
        chapter_node = T_bj.nodeFromSection((book_code, chapter_num))
        verses = L_bj.d(chapter_node, otype='verse')
        lines = []
        for v_node in verses:
            words = L_bj.d(v_node, otype='word')
            text = ' '.join(F_bj.text.v(w) for w in words if F_bj.text.v(w))
            ref = T_bj.sectionFromNode(v_node)
            lines.append(f"**{ref[2]}.** {text}")
        return '\n'.join(lines)

    print(f"=== Mark ch.{max_ch} — realis score: {df_mark.loc[df_mark.chapter == max_ch, 'net_score'].values[0]:.3f} ===")
    print(get_bj_chapter_text(BJ_BOOK, max_ch))
    print()
    print(f"=== Mark ch.{min_ch} — realis score: {df_mark.loc[df_mark.chapter == min_ch, 'net_score'].values[0]:.3f} ===")
    print(get_bj_chapter_text(BJ_BOOK, min_ch))

except Exception as e:
    print(f"BJ dataset not available: {e}")
    print(f"Expected path: {BJ_PATH}")

## 9. John's Gospel: The Farewell Discourse Anomaly

Among the four gospels, **John's arc has a unique shape**: it dips sharply in the middle —
not because the narrative slows down, but because a massive block of sustained *discourse*
displaces narrative action entirely.

### What is the Farewell Discourse?

Chapters 13–17 of John are known as the **Farewell Discourse** (or *Discours d'adieu*):
five consecutive chapters spoken by Jesus to his disciples at the Last Supper, the night before
his arrest. There is almost no action. Instead: intimate teaching, prayer, exhortation,
promise, comfort. Linguistically, this is a discourse in the most technical sense —
a sustained monologue dense with subjunctives (*"that you may believe"*, *"so that your joy
may be full"*), imperatives (*"love one another"*, *"do not let your hearts be troubled"*),
and conditional constructions (*"if you love me"*, *"if the world hates you"*).

This is precisely what our morphological scoring captures: **irrealis forms dominate**
because the author is not reporting events but articulating obligations, possibilities,
and desired states.

### Why does this matter?

The contrast is stark enough to be detectable from morphology alone — with **zero semantic
analysis**, no named-entity recognition, no dialogue detection:

- John 1–12 (*"The Book of Signs"*): miracles, healings, confrontations — high realis
- John 13–17 (*"The Book of Glory"*, discourse section): the deepest dip in all four gospels
- John 18–21 (*Passion and Resurrection*): realis rebounds as event narration returns

This is a **computationally measurable literary structure** that scholars have identified
for centuries through close reading. The fact that raw morphological weights reproduce it
validates the morpho-aspectual approach.

In [ ]:
df_john = dfs["John"]

# Define the three structural sections of John
SECTIONS = {
    "Book of Signs (ch. 1–12)":          range(1, 13),
    "Farewell Discourse (ch. 13–17)":    range(13, 18),
    "Passion & Resurrection (ch. 18–21)": range(18, 22),
}

print("John — Net Realis Score by Section")
print("=" * 45)
for section_name, ch_range in SECTIONS.items():
    section_df = df_john[df_john["chapter"].isin(ch_range)]
    mean_score    = section_df["net_score"].mean()
    mean_irrealis = section_df["irrealis_pct"].mean()
    mean_strong   = section_df["realis_strong_pct"].mean()
    print(f"\n{section_name}")
    print(f"  Mean net realis score : {mean_score:.3f}")
    print(f"  Mean strong realis %  : {mean_strong:.1f}%")
    print(f"  Mean irrealis %       : {mean_irrealis:.1f}%")

# ── Focused chart: John chapter-by-chapter with section shading ──
fig, ax = plt.subplots(figsize=(13, 5))

john_chapters = df_john["chapter"].tolist()
john_scores   = df_john["net_score"].values
john_smoothed = savgol_filter(john_scores, window_length=3, polyorder=2)

# Section background shading
ax.axvspan(1,  12.5, alpha=0.06, color="#1a4f72", label="Book of Signs (ch. 1–12)")
ax.axvspan(12.5, 17.5, alpha=0.12, color="#e07b39", label="Farewell Discourse (ch. 13–17)")
ax.axvspan(17.5, 21,  alpha=0.06, color="#2e8b57", label="Passion & Resurrection (ch. 18–21)")

# Raw and smoothed arc
ax.plot(john_chapters, john_scores,
        color="#8b2252", linewidth=1, linestyle="--", alpha=0.4)
ax.plot(john_chapters, john_smoothed,
        color="#8b2252", linewidth=2.5, label="John — smoothed arc")

# Annotate the trough
trough_ch = int(df_john.loc[df_john.net_score.idxmin(), "chapter"])
ax.annotate(
    f"Ch.{trough_ch}\n(discourse minimum)",
    xy=(trough_ch, john_smoothed[trough_ch - 1]),
    xytext=(trough_ch + 0.6, john_smoothed[trough_ch - 1] - 0.04),
    fontsize=9, arrowprops=dict(arrowstyle="->", color="#333")
)

ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
ax.set_xlabel("Chapter", fontsize=12)
ax.set_ylabel("Net realis score (mean weight)", fontsize=12)
ax.set_title(
    "John — Narrative Arc with Farewell Discourse Highlighted\n(Greek Verbal Morphology, N1904)",
    fontsize=13
)
ax.set_xticks(john_chapters)
ax.legend(fontsize=9, loc="lower left")
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

# ── Stacked bar for John, same palette as Mark ──
fig2, ax2 = plt.subplots(figsize=(13, 5))
bottom2 = [0.0] * len(john_chapters)

for col, label, color in zip(STACK_COLS, STACK_LABELS, COLORS.values()):
    vals = df_john[col].tolist()
    ax2.bar(john_chapters, vals, bottom=bottom2, label=label, color=color, width=0.8)
    bottom2 = [b + v for b, v in zip(bottom2, vals)]

# Highlight the Farewell Discourse block
for ch in range(13, 18):
    ax2.axvline(ch - 0.4, color="#e07b39", linewidth=0.5, alpha=0.4)
    ax2.axvline(ch + 0.4, color="#e07b39", linewidth=0.5, alpha=0.4)
ax2.axvspan(12.6, 17.4, alpha=0.08, color="#e07b39", label="Farewell Discourse (ch. 13–17)")

ax2.set_xlabel("Chapter", fontsize=12)
ax2.set_ylabel("% of classified verbs", fontsize=12)
ax2.set_title("John — Verbal Modal Composition by Chapter", fontsize=13)
ax2.set_xticks(john_chapters)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()